# Exploración ligera de datos
Ejecute después de preparar el proyecto y los datos. Solo lee parquets y el manifiesto; no entrena modelos. Los cálculos de retornos se restringen al periodo de desarrollo. La calidad registrada en el manifiesto puede describir el archivo completo, incluido el periodo reservado, y se muestra únicamente como metadatos.

In [ ]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd

REPO_DIR = Path("/content/quant-ai") if Path("/content/quant-ai/configs/base.yaml").exists() else Path.cwd()
if not (REPO_DIR / "configs/base.yaml").exists() and (REPO_DIR.parent / "configs/base.yaml").exists():
    REPO_DIR = REPO_DIR.parent
from quant_ai.config import load_config
DATA_HOME = Path("/content/drive/MyDrive/quant-ai") if str(REPO_DIR).startswith("/content/") else REPO_DIR
# En Colab monte Drive antes de abrir este notebook en una sesión nueva.
os.environ.setdefault("QUANT_AI_HOME", str(DATA_HOME))
cfg = load_config(REPO_DIR / "configs/base.yaml")
DATA_DIR = cfg.data_dir
manifest_path = DATA_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
if manifest.get("latest"):
    display(pd.json_normalize([dict(symbol=s, **v) for s, v in manifest["latest"].items()]))
else:
    print("No hay resumen de calidad en data/manifest.json.")

In [ ]:
frames, quality = {}, []
for path in sorted((DATA_DIR / "clean").glob("*.parquet")):
    if path.stem not in {f"{s}_{cfg.interval}" for s in cfg.symbols}:
        continue
    symbol = path.stem.rsplit("_", 1)[0]
    frame = pd.read_parquet(path, columns=["open_time", "close", "volume"])
    frame["open_time"] = pd.to_datetime(frame.open_time, utc=True)
    frame = frame.loc[frame.open_time < cfg.final_holdout_start].sort_values("open_time")
    quality.append({"symbol": symbol, "filas_desarrollo": len(frame),
                    "inicio": frame.open_time.min(), "fin": frame.open_time.max(),
                    "duplicadas": int(frame.open_time.duplicated().sum()),
                    "saltos_no_horarios": int(frame.open_time.diff().iloc[1:].ne(pd.Timedelta("1h")).sum()),
                    "close_nulos": int(frame.close.isna().sum()),
                    "close_no_positivos": int(frame.close.le(0).sum())})
    if frame.open_time.duplicated().any():
        raise ValueError(f"{symbol}: open_time duplicado; revise el proceso de limpieza.")
    frames[symbol] = frame.set_index("open_time")
if not frames:
    raise FileNotFoundError(f"No hay parquets de los símbolos configurados en {DATA_DIR / 'clean'}")
display(pd.DataFrame(quality))

## Retornos y volatilidad
Se anulan retornos que crucen huecos. La volatilidad móvil usa 168 horas y se anualiza con √8760; es una descripción histórica, no un pronóstico.

In [ ]:
returns = {}
for symbol, frame in frames.items():
    hourly = frame.close.where(frame.close > 0).resample("h").asfreq()
    returns[symbol] = np.log(hourly).diff()
returns = pd.DataFrame(returns)
display(returns.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)
volatility = returns.rolling(168, min_periods=168).std() * np.sqrt(8760)
display(volatility.resample("ME").last().tail(24))
display(pd.DataFrame({"vol_anualizada": returns.std() * np.sqrt(8760),
                      "retorno_medio_hora": returns.mean(),
                      "observaciones": returns.count()}))

## Autocorrelación y relación BTC–ETH
Los rezagos se calculan sobre la cuadrícula horaria sin rellenar precios faltantes. La correlación usa solo observaciones simultáneas válidas.

In [ ]:
autocorrelation = pd.DataFrame({symbol: {lag: series.autocorr(lag) for lag in [1, 2, 6, 12, 24, 168]}
                                for symbol, series in returns.items()}).rename_axis("rezago_horas")
display(autocorrelation)
display(returns.corr())
if {"BTCUSDT", "ETHUSDT"}.issubset(returns.columns):
    rolling_correlation = returns.BTCUSDT.rolling(168, min_periods=168).corr(returns.ETHUSDT)
    display(rolling_correlation.resample("ME").last().tail(24).rename("correlacion_BTC_ETH"))